In [11]:
!pip install transformers torch

In [12]:
# importing required libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
import torch

# loading model
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

# memory storage (simple dictionary)
memory = {}

chat_history_ids = None
MAX_HISTORY = 1000

# predefined responses
greetings = ["hi", "hello", "hey"]
thanks = ["thanks", "thank you"]

print("Assistant: Hello! I’m your AI assistant. Ask me anything 😊")

while True:
    try:
        user_input = input("You: ").strip()

        # exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Assistant: Goodbye! Have a great day 👋")
            break

        lower_input = user_input.lower()

        # ---------------- MEMORY FEATURE ----------------
        if "my name is" in lower_input:
            name = user_input.split("is")[-1].strip()
            memory["name"] = name
            print(f"Assistant: Nice to meet you, {name}!")
            continue

        if "what is my name" in lower_input:
            if "name" in memory:
                print(f"Assistant: Your name is {memory['name']} 😊")
            else:
                print("Assistant: I don't know your name yet. Tell me!")
            continue

        # ---------------- TASK HANDLING ----------------
        if "time" in lower_input:
            from datetime import datetime
            current_time = datetime.now().strftime("%H:%M:%S")
            print(f"Assistant: Current time is {current_time}")
            continue

        if "date" in lower_input:
            from datetime import datetime
            today = datetime.now().strftime("%Y-%m-%d")
            print(f"Assistant: Today's date is {today}")
            continue

        # ---------------- GREETING ----------------
        if lower_input in greetings:
            print("Assistant: Hello! How can I assist you today?")
            continue

        if lower_input in thanks:
            print("Assistant: You're welcome! 😊")
            continue

        # ---------------- AI RESPONSE ----------------
        new_input_ids = tokenizer.encode(
            user_input + tokenizer.eos_token,
            return_tensors='pt'
        )

        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids

        bot_input_ids = bot_input_ids[:, -MAX_HISTORY:]
        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=bot_input_ids.shape[-1] + 60,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=40,
            top_p=0.9,
            temperature=0.6,
            no_repeat_ngram_size=3
        )

        response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        ).strip()

        # fallback safety
        if len(response.split()) < 2:
            response = "I'm not fully sure, but I can help if you explain a bit more."

        print("Assistant:", response)

    except Exception:
        print("Assistant: Something went wrong. Please try again.")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Assistant: Hello! I’m your AI assistant. Ask me anything 😊
You: Exit
Assistant: Goodbye! Have a great day 👋
